# SACDpy batch timelapse z-stack reconstruction

Canonical pipeline for SACD live-cell acquisitions stored as discrete time-indexed z-stack TIFF movies. It combines recursive dataset discovery, explicit FOV selection, pilot execution, resumable batching, z-MIP generation, validation, provenance, and live ETA reporting.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import tifffile

repo = Path.cwd()
src = repo / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from sacdpy.continuous_batch import (
    build_batch_plan, load_config, preflight_summary, run_batch, validate_output_pair,
)
print(f'Working folder: {repo}')

## 2. Configuration

The JSON file is the reproducible source of paths, FOV selections/exclusions, aliases, and SACD parameters. To process only specific FOVs, add their dataset-relative paths to `selected_fov_folders` for that dataset.

In [ ]:
config_path = repo / 'configs' / 'sacd_live_batch_2026.json'
config = load_config(config_path)
print(json.dumps(config, indent=2))

## 3. Read-only discovery and preflight

Review every accepted/excluded FOV and planned output before writing results.

In [ ]:
plan = build_batch_plan(config)
summary = preflight_summary(plan)
print(json.dumps(summary, indent=2))
print('\nAccepted FOVs:')
for fov in plan.fovs:
    print(f"+ {fov.dataset_name}/{fov.relative_fov}: T={len(fov.group.time_indices)}, Z={len(fov.group.z_indices)}, movies={fov.movie_count}")
    print(f"  -> {fov.stack_output.name}; {fov.mip_output.name}")
print('\nExcluded FOVs:')
for item in plan.exclusions:
    print(f"- {item['dataset_name']}/{item['relative_fov']}: {item['reason']}")

## 4. Pilot one new FOV

This is resumable. A subsequent full run recognizes the validated pilot in its manifest and continues with the next FOV.

In [ ]:
pilot_results = run_batch(config, config_path, max_new_fovs=1)
pilot_result = next(item for item in reversed(pilot_results) if item['status'] in {'written', 'skipped_existing'})
print(json.dumps(pilot_result, indent=2, default=str))

## 5. Pilot preview

In [ ]:
validate_output_pair(pilot_result['stack_output'], pilot_result['mip_output'])
stack = tifffile.imread(pilot_result['stack_output'])
mip = tifffile.imread(pilot_result['mip_output'])
fig, axes = plt.subplots(1, stack.shape[1] + 1, figsize=(4 * (stack.shape[1] + 1), 4), squeeze=False)
for z_index, ax in enumerate(axes.ravel()[:-1]):
    ax.imshow(stack[0, z_index], cmap='hot')
    ax.set_title(f't0 z{z_index}')
    ax.axis('off')
axes.ravel()[-1].imshow(mip[0], cmap='hot')
axes.ravel()[-1].set_title('t0 SACD MIP')
axes.ravel()[-1].axis('off')
plt.tight_layout();

## 6. Full resumable batch

Run after pilot inspection. `run_status.json`, `manifest.json`, and `run.log` are updated after each FOV.

In [ ]:
results = run_batch(config, config_path)
print(f'Run returned {len(results)} result record(s).')

## 7. Final summary

In [ ]:
for dataset in config['datasets']:
    result_dir = Path(config['output_root']) / dataset.get('result_name', Path(dataset['raw_root']).name)
    manifest_path = result_dir / '_processing' / 'manifest.json'
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        statuses = {}
        for item in manifest.get('fovs', []):
            statuses[item['status']] = statuses.get(item['status'], 0) + 1
        print(result_dir.name, statuses, f"existing_outputs={len(manifest.get('existing_outputs', []))}")